# FRED Data - Exploratory Data Analysis
Exploring the cleaned risk-free rate and CPI data before it feeds into the Sharpe ratio and backtest calculations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('default')
%matplotlib inline

In [ ]:
PROCESSED_DATA_PATH = Path('../data/processed')

## Load processed data

In [ ]:
rf = pd.read_csv(PROCESSED_DATA_PATH / 'fred_risk_free_rate_processed.csv', index_col='date', parse_dates=True)
cpi = pd.read_csv(PROCESSED_DATA_PATH / 'fred_cpi_processed.csv', index_col='date', parse_dates=True)

print('Risk-free rate shape:', rf.shape)
print('CPI shape:', cpi.shape)

## Basic structure and summary stats

In [ ]:
rf.info()

In [ ]:
rf.describe()

In [ ]:
cpi.describe()

## Missing values check
Should be zero at this point - if not, go back to the preprocessing notebook.

In [ ]:
print('Risk-free rate missing values:')
print(rf.isna().sum())
print()
print('CPI missing values:')
print(cpi.isna().sum())

## Risk-free rate over time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rf.index, rf['risk_free_rate_pct'], linewidth=1)
ax.set_title('3-Month Treasury Yield (Risk-Free Rate) Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Rate (%)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Distribution of the risk-free rate

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(rf['risk_free_rate_pct'], bins=40, edgecolor='black', alpha=0.7)
ax.set_title('Distribution of Risk-Free Rate')
ax.set_xlabel('Rate (%)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## CPI over time and inflation rate

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(cpi.index, cpi['cpi_index'], linewidth=1, color='darkorange')
axes[0].set_title('CPI Index Over Time')
axes[0].set_ylabel('CPI Index')
axes[0].grid(alpha=0.3)

axes[1].plot(cpi.index, cpi['cpi_pct_change'] * 100, linewidth=1, color='green')
axes[1].set_title('CPI Percent Change (Daily-Interpolated)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('% Change')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Year-over-year summary
Quick sanity check on how the risk-free rate has moved by year - useful for spotting whether your backtest window includes any unusual rate regimes (e.g. near-zero rates, rapid hikes).

In [ ]:
rf_yearly = rf['risk_free_rate_pct'].resample('YE').agg(['mean', 'min', 'max'])
rf_yearly.columns = ['avg_rate', 'min_rate', 'max_rate']
rf_yearly

## Trim to backtest date range
Replace these placeholder dates with your team's actual backtest start/end dates once confirmed.

In [ ]:
BACKTEST_START = '2015-01-01'  # TODO: update to match team's actual backtest window
BACKTEST_END = '2024-12-31'    # TODO: update to match team's actual backtest window

rf_backtest = rf.loc[BACKTEST_START:BACKTEST_END]
cpi_backtest = cpi.loc[BACKTEST_START:BACKTEST_END]

print(f'Risk-free rate rows in backtest window: {len(rf_backtest)}')
print(f'CPI rows in backtest window: {len(cpi_backtest)}')

rf_backtest['risk_free_rate_pct'].plot(figsize=(12,4), title='Risk-Free Rate - Backtest Window Only')
plt.tight_layout()
plt.show()

## Key takeaways
Summarize what you found here once you've run through the notebook - e.g. rate range during the backtest period, any gaps or anomalies, average risk-free rate to use as a baseline for Sharpe ratio calculations.